# 第三课｜“都叫 LIF”为什么还不够？

前两课我们已经有了一个可运行的 **漏电积分发放模型（Leaky Integrate-and-Fire, LIF）**，也看到有限位宽数值表示会改变模型轨迹。

现在出现一个非常工程化、但也非常科学的问题：

> **如果 Python、AI 和未来的硬件对“同一个 LIF”有不同理解，谁才算正确？**

本课的主要新概念：**实现之前，要先冻结可测试的模型语义。**

## 1. 先认识三个词：specification、semantics、test oracle

### Specification：规范
**规范（specification，常简称 spec）** 是我们明确写下来的“系统必须怎样工作”的规则。它不是代码本身，而是代码应该满足的契约。

### Semantics：语义
**语义（semantics）** 在这里指一条规则到底是什么意思。比如“达到阈值时放电”听起来很清楚，但到底是 `V > threshold` 还是 `V >= threshold`？这就是语义细节。

### Test oracle：测试真值依据
**测试真值依据（test oracle）** 是我们判断测试结果对不对时所依赖的权威标准。

如果 Python 和硬件结果不同，我们不能简单说“Python 一定对”或者“硬件一定对”。真正有权决定的是已经冻结的模型规范，以及由它产生的参考测试。

## 2. 一个字符，就可能变成两个不同模型

假设更新后的膜电位刚好等于 threshold：

- 规则 A：`new_v >= threshold` 时 spike；
- 规则 B：`new_v > threshold` 时 spike。

这两个写法只差一个等号，但行为不同。

让我们构造一个故意卡在边界上的例子。

In [ ]:
def step_ge(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v >= threshold
    return (reset if spike else new_v), spike

def step_gt(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v > threshold
    return (reset if spike else new_v), spike

v0 = 0.75
current = 0.25

print('>= rule:', step_ge(v0, current))
print(' > rule:', step_gt(v0, current))

## 3. Observe：为什么这个实验比“随机测试很多次”更有价值？

因为这是一个**边界测试（boundary test）**：我们故意把输入放在两个定义最容易分叉的位置。

在普通随机输入下，`>` 和 `>=` 可能很久都表现一样；但这不代表它们语义相同。

这也是测试设计的重要直觉：

> 好测试不是只追求数量多，而是要主动寻找能区分不同解释的输入。

## 4. 另一个歧义：先衰减再加输入，还是先加输入再衰减？

考虑下面两个更新规则：

A. `new_v = alpha * v + input`

B. `new_v = alpha * (v + input)`

它们看起来都可以被口头描述成“有 leak，也会 integrate input”，但数值并不相同。

In [ ]:
v = 0.8
current = 0.4
alpha = 0.9

rule_a = alpha * v + current
rule_b = alpha * (v + current)

print('A: leak old state, then add input ->', rule_a)
print('B: add input, then leak everything  ->', rule_b)

## 5. “常见写法”不能替代我们的规范

你在论文、教材或网上可能会看到很多不同版本的 LIF：

- 有的使用连续时间微分方程；
- 有的使用离散时间更新；
- 有的 threshold 用 `>`，有的用 `>=`；
- 有的 spike 后立刻 reset，有的在下一步 reset；
- 有的 refractory period 期间忽略输入，有的继续积累；
- 有的加入噪声或 tonic drive。

它们都可能有合理用途。

所以工程上不能写一句“我们用标准 LIF”就结束。我们必须说明**我们的 LIF 到底是哪一个版本**。

## 6. 我们需要冻结哪些语义？

至少要逐项回答：

| 问题 | 为什么必须说清楚 |
|---|---|
| 状态 `V` 在什么时候读取？ | 决定本步计算使用旧状态还是新状态 |
| leak 和 input 的顺序是什么？ | 不同顺序可能得到不同数值 |
| threshold 用 `>` 还是 `>=`？ | 边界点 spike 行为不同 |
| spike 后 reset 到多少？ | 决定下一步初始状态 |
| refractory period 多久？ | 决定之后哪些步允许 spike |
| refractory 期间是否接受输入？ | 决定输入是否丢失/积累 |
| rounding 在哪一步发生？ | fixed-point 中会改变轨迹 |
| overflow 怎样处理？ | saturation 与 wraparound 行为完全不同 |
| 同一 step 多个输入如何合并？ | 决定累积顺序和并发语义 |

这张表以后会变成真正的 model contract。

## 7. 什么叫“冻结”语义？

“冻结”不是说以后永远不能改。它的意思是：

1. 当前版本先明确选择一种规则；
2. 给这个版本一个可追踪的定义；
3. 写出能区分关键语义的测试；
4. Python、未来的硬件和测试都引用同一份定义；
5. 如果以后要改，必须把它当成版本变化，而不是偷偷在某段代码里改。

这让我们可以回答一句非常重要的话：

> **现在这个模型为什么是对的？因为它符合我们已经明确写下并测试过的规范。**

In [ ]:
semantic_decisions = {
    'threshold_comparison': 'TBD: >= or >',
    'update_order': 'TBD: leak_then_input or input_then_leak',
    'reset_value': 'TBD',
    'refractory_steps': 'TBD',
    'input_during_refractory': 'TBD',
    'rounding_rule': 'TBD',
    'overflow_rule': 'TBD',
    'same_step_input_accumulation': 'TBD',
}

for key, value in semantic_decisions.items():
    print(f'{key:30s} {value}')

## 8. Try It：为每个歧义设计一个最小反例

不要急着填 `TBD`。先选其中两个问题，分别设计一个输入，使两种候选规则一定会给出不同结果。

例如：

- `>` vs `>=`：让 candidate voltage 恰好等于 threshold；
- saturation vs wraparound：故意让结果超过最大可表示值；
- refractory 是否积累输入：在 refractory 期间给一个大输入，然后看恢复后状态。

如果你找不到能区分两种规则的测试，说明你可能还没有真正理解这两个规则的区别。

## 9. AI Task

可以让 AI 帮你生成一份“LIF 实现前必须明确的歧义清单”，但要求每一项同时给出：

1. 两种候选语义；
2. 一个最小区分测试；
3. 可能影响 spike sequence 的原因。

然后由人决定哪些语义进入 v0 规范。

AI 可以帮助我们发现遗漏，但不能因为“常见实现一般这样写”就替项目做无记录的决定。

## 10. Human Check

不用 AI，你应该能回答：

- specification、semantics、test oracle 分别是什么？
- 为什么 `>` 和 `>=` 的差别不是“代码风格”？
- 为什么随机测试全部通过，仍可能存在边界语义错误？
- 为什么 Python reference 也必须受 spec 约束，而不是天然拥有最高权威？
- 如果以后想换一种 LIF 语义，正确做法是什么？

## 11. Engineering Handoff

本课的产物不是更多算法代码，而是一个 **spec checkpoint**：

- 把 v0 神经元语义写入 MDD；
- 把边界测试和 oracle 写入 TDD；
- 把需求、设计、测试、实现任务的对应关系写入 TRACE。

未来我们会使用 **寄存器传输级（Register-Transfer Level, RTL）** 来描述数字硬件。RTL 可以理解成“描述哪些状态由寄存器保存，以及数据在时钟周期之间怎样计算和传递”的设计层次。今天只需要认识这个词；下一阶段才正式学习。

## 12. 项目追踪 Project Trace

- Lesson ID: `LSN-003`
- Engineering slice: `RMD-003`
- Process requirements: `PFR3 / PFR4`
- Process design: `PDP3 / PDP4`

这些 ID 只用于长期追踪，不是本课的学习负担。

## 13. Exit Ticket

进入下一课前：

1. 你能解释 specification、semantics 和 test oracle；
2. 你能举出至少两个“都叫 LIF，但语义不同”的例子；
3. `semantic_decisions` 中影响实现的项目不再是无意识的默认值；
4. 每个关键决定至少有一个能故意让错误实现失败的测试。

下一课我们第一次真正靠近数字硬件：

> 软件变量可以自然地保存 `v`；一块电路要怎样记住上一时刻的 `v`？